# Загрузка библиотек

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
import pickle

# Загрузка данных

In [2]:
filepath = "data/Mall_Customers.csv" # Путь до файла
sep = "," # Разделитель данных (открой файл и посмотри)

data = pd.read_csv(filepath, sep=sep)

data.head()

,CustomerID,Gender,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


# Загрузка данных в базу данных

In [3]:
def get_conn(dbname, user, password, host):
    url = f"postgresql+psycopg2://{user}:{password}@{host}/{dbname}"
    return create_engine(url)

In [21]:
dbname = "Mall_Customers" # Название базы данных
user = "postgres" # Имя пользователя для подключения
password = "86754231qaZ" # Пароль пользователя для подключения
host = "localhost" # Хост


conn = get_conn(dbname, user, password, host)

In [ ]:
query = """
DROP TABLE mall_customers;

CREATE TABLE mall_customers
(
    CustomerID SERIAL PRIMARY KEY,
    Gender VARCHAR(255),
    Age SMALLINT,
    "Annual Income (k$)" INTEGER,
    "Spending Score (1-100)" SMALLINT
);
"""

insert_query = """
INSERT INTO mall_customers (
CustomerID, Gender, Age, "Annual Income (k$)", "Spending Score (1-100)"
) VALUES (%s, %s, %s, %s, %s);
"""

data_for_db = list(data.itertuples(index=False, name=None))

with conn.connect() as conn:
    conn.execute(query, data_for_db)
    conn.commit()

ProgrammingError: (psycopg2.errors.SyntaxError) syntax error at or near ","
LINE 15: ) VALUES (?, ?, ?, ?, ?);
                    ^

[SQL: 
DROP TABLE mall_customers;

CREATE TABLE mall_customers
(
    CustomerID SERIAL PRIMARY KEY,
    Gender VARCHAR(255),
    Age SMALLINT,
    "Annual Income (k$)" INTEGER,
    "Spending Score (1-100)" SMALLINT
);

INSERT INTO mall_customers (
CustomerID, Gender, Age, "Annual Income (k$)", "Spending Score (1-100)"
) VALUES (?, ?, ?, ?, ?);
]
[parameters: [(1, 'Male', 19, 15, 39), (2, 'Male', 21, 15, 81), (3, 'Female', 20, 16, 6), (4, 'Female', 23, 16, 77), (5, 'Female', 31, 17, 40), (6, 'Female', 22, 17, 76), (7, 'Female', 35, 18, 6), (8, 'Female', 23, 18, 94)  ... displaying 10 of 200 total bound parameter sets ...  (199, 'Male', 32, 137, 18), (200, 'Male', 30, 137, 83)]]
(Background on this error at: https://sqlalche.me/e/14/f405)

In [9]:
data.to_sql("mall_customers", conn, index=False, if_exists="replace")

/tmp/ipykernel_12603/3039996389.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data.to_sql("mall_customers", conn, index=False, if_exists="replace")


DatabaseError: Execution failed on sql '
        SELECT
            name
        FROM
            sqlite_master
        WHERE
            type IN ('table', 'view')
            AND name=?;
        ': syntax error at or near ";"
LINE 8:             AND name=?;
                              ^


# Предобработка данных

In [56]:
cat_cols = ["Gender"] # Категориальные признаки

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop="first")

# Применяем к категориальным колонкам
encoded_data = encoder.fit_transform(data[cat_cols])

# Получаем имена новых колонок
feature_names = encoder.get_feature_names_out(cat_cols)

# Создаем DataFrame с закодированными данными
data_encoded = pd.DataFrame(encoded_data, columns=feature_names)

# Объединяем с числовыми колонками
data = pd.concat([data.drop(cat_cols, axis=1), data_encoded], axis=1)

In [57]:
data.head()

,CustomerID,Age,Annual Income (k$),Spending Score (1-100),Gender_Male
0,1,19,15,39,1.0
1,2,21,15,81,1.0
2,3,20,16,6,0.0
3,4,23,16,77,0.0
4,5,31,17,40,0.0


In [58]:
pickle.dump(encoder, open("models/Encoder.sav", 'wb'))

In [59]:
data = data.drop(columns=["CustomerID"])

# Обучение модели

In [60]:
data

,Age,Annual Income (k$),Spending Score (1-100),Gender_Male
0,19,15,39,1.0
1,21,15,81,1.0
2,20,16,6,0.0
3,23,16,77,0.0
4,31,17,40,0.0
...,...,...,...,...
195,35,120,79,0.0
196,45,126,28,0.0
197,32,126,74,1.0
198,32,137,18,1.0


In [61]:
X = data.drop(columns=["Spending Score (1-100)"])
Y = data["Spending Score (1-100)"]

In [62]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [63]:
pickle.dump(scaler, open("models/Scaler.sav", 'wb'))

In [64]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, Y, test_size=0.2, random_state=42)

In [65]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

In [66]:
print("Коэффициенты линейной регрессии:")
for feature, coef in zip(X.columns, lr.coef_):
    print(f"{feature}: {coef:.4f}")
print(f"\nСвободный член (intercept): {lr.intercept_:.4f}")

Коэффициенты линейной регрессии:
Age: -8.1922
Annual Income (k$): 1.4112
Gender_Male: -0.5949

Свободный член (intercept): 51.7988


In [67]:
print("RMSE:", root_mean_squared_error(y_test, y_pred))

RMSE: 21.924259205438346


In [68]:
pickle.dump(lr, open("models/Regression_Model.sav", 'wb'))